# PLG Data Preparation — Creating User-Level Summary for Feature Adoption & Conversion Analysis


This notebook prepares the clean, user-level dataset used for Product-Led Growth (PLG) analytics.

We load the raw SaaS subscription dataset (accounts, subscriptions, feature usage, churn events, support tickets), engineer feature usage flags, compute upgrade and churn labels, and generate the final `user_plg_summary.csv`.

This dataset is later used in Tableau to build:
- User journey funnel
- Feature adoption vs conversion analysis
- Power-user effect
- Feature adoption distribution


In [6]:
import pandas as pd


In [7]:
accounts = pd.read_csv("/kaggle/input/datasets/purvashah48/plg-saas-subscription-analysis/accounts.csv")
subscriptions = pd.read_csv("/kaggle/input/datasets/purvashah48/plg-saas-subscription-analysis/subscriptions.csv")
feature_usage = pd.read_csv("/kaggle/input/datasets/purvashah48/plg-saas-subscription-analysis/feature_usage.csv")
churn = pd.read_csv("/kaggle/input/datasets/purvashah48/plg-saas-subscription-analysis/churn_events.csv")
tickets = pd.read_csv("/kaggle/input/datasets/purvashah48/plg-saas-subscription-analysis/support_tickets.csv")


## 1. Prepare Feature Usage Flags
We pivot feature usage into binary feature flags and compute total events per account.


In [8]:
# Join feature_usage with subscriptions to get account_id
feature_usage_with_account = feature_usage.merge(
    subscriptions[["subscription_id", "account_id"]],
    on="subscription_id"
)

# Pivot to create feature flags
feature_pivot = (
    feature_usage_with_account
    .pivot_table(
        index="account_id",
        columns="feature_name",
        values="usage_count",
        aggfunc="sum"
    )
    .fillna(0)
)

# Rename known features to readable names
feature_pivot = feature_pivot.rename(columns={
    "invite_teammate": "did_invite_teammate",
    "create_project": "did_create_project",
    "connect_integration": "did_connect_integration",
    "export_report": "did_export_report"
}, errors="ignore")

# Convert counts → binary flags
for col in feature_pivot.columns:
    feature_pivot[col] = (feature_pivot[col] > 0).astype(int)

# Add total events
feature_pivot["total_events"] = (
    feature_usage_with_account.groupby("account_id")["usage_count"].sum()
)


## 2. Prepare Subscription Upgrade Labels


In [9]:
subscriptions["upgraded"] = subscriptions["upgrade_flag"].astype(int)
subscription_summary = subscriptions.groupby("account_id")["upgraded"].max()


## 3. Prepare Churn Labels


In [10]:
churn["churned"] = churn["churn_date"].notna().astype(int)
churn_summary = churn.groupby("account_id")["churned"].max()


## 4. Prepare Support Ticket Summary


In [11]:
ticket_summary = tickets.groupby("account_id").agg(
    total_support_tickets=("ticket_id", "count"),
    avg_resolution_time=("resolution_time_hours", "mean")
)


## 5. Merge All Components into User-Level Summary


In [12]:
summary = (
    accounts.set_index("account_id")
    .join(feature_pivot, how="left")
    .join(subscription_summary, how="left")
    .join(churn_summary, how="left")
    .join(ticket_summary, how="left")
)

# Fill missing values
summary = summary.fillna({
    "did_invite_teammate": 0,
    "did_create_project": 0,
    "did_connect_integration": 0,
    "did_export_report": 0,
    "total_events": 0,
    "upgraded": 0,
    "churned": 0,
    "total_support_tickets": 0
})


## 6. Save Final User-Level Summary


In [13]:
summary.to_csv("user_plg_summary.csv")
summary.head()


,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag,feature_1,...,feature_5,feature_6,feature_7,feature_8,feature_9,total_events,upgraded,churned,total_support_tickets,avg_resolution_time
account_id,,,,,,,,,,,,,,,,,,,,,
A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,False,False,1,...,1,1,1,1,0,535,1,1.0,2.0,23.000000
A-43a9e3,Company_1,FinTech,IN,2023-08-17,other,Basic,18,False,True,1,...,1,0,0,1,1,355,1,0.0,3.0,38.000000
A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,False,False,1,...,1,1,1,1,1,821,1,1.0,3.0,43.666667
A-1f0ac7,Company_3,HealthTech,UK,2023-08-27,other,Basic,24,True,False,1,...,0,1,1,0,1,382,1,1.0,2.0,29.000000
A-ce550d,Company_4,HealthTech,US,2024-10-27,event,Enterprise,35,False,True,1,...,0,1,0,1,1,579,1,1.0,7.0,42.285714


### ✔ Output Generated
`user_plg_summary.csv` is now ready for:

- Tableau dashboard
- Feature adoption analysis
- Conversion modeling
- PLG insights

This notebook focuses only on **data preparation**.  
All visualizations are created in Tableau.
